Dummy Classifier (Baseline)

This notebook implements a baseline model that makes predictions using simple strategies (like always picking the most frequent class). This serves as a point of comparison for all other models.

In [1]:
import os
import pickle

import mlflow
import mlflow.sklearn
import pandas as pd
from metrics_utils import calculate_business_metrics
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_experiment("Walmart-Sales-Classification")

Traceback (most recent call last):
  File "/home/sarah/.cache/pypoetry/virtualenvs/walmart-sales-classification-0W0ZSQ-Z-py3.12/lib/python3.12/site-packages/mlflow/store/tracking/file_store.py", line 329, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sarah/.cache/pypoetry/virtualenvs/walmart-sales-classification-0W0ZSQ-Z-py3.12/lib/python3.12/site-packages/mlflow/store/tracking/file_store.py", line 427, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sarah/.cache/pypoetry/virtualenvs/walmart-sales-classification-0W0ZSQ-Z-py3.12/lib/python3.12/site-packages/mlflow/store/tracking/file_store.py", line 1373, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  F

<Experiment: artifact_location='file:///home/sarah/code/forth_year/data_science/project/walmart-sales-classification/src/models/mlruns/810253398722391848', creation_time=1777489467114, experiment_id='810253398722391848', last_update_time=1777489467114, lifecycle_stage='active', name='Walmart-Sales-Classification', tags={}>

In [2]:
train_df = pd.read_csv("../../data/model_ready/train.csv")
test_df = pd.read_csv("../../data/model_ready/test.csv")

features_selected = [
    "Size",
    "Store",
    "Dept",
    "CPI",
    "DeptFrequency",
    "Week_cos",
    "IsPreHoliday",
    "Week_sin",
    "Fuel_Price",
    "ConsumerConfRatio",
    "AvgMarkDownAmount",
]
target = "Sales_Class"
holiday_col = "IsHoliday"

X_train = train_df[features_selected]
y_train = train_df[target]
X_test = test_df[features_selected]
y_test = test_df[target]

# We need IsHoliday for business metrics evaluation
test_holidays = test_df[holiday_col]

In [3]:
with mlflow.start_run(run_name="Dummy_Baseline"):
    model_path = "dummy_model.pkl"
    if os.path.exists(model_path):
        with open(model_path, "rb") as f:
            model = pickle.load(f)
        print("Model loaded from pickle")
    else:
        # 1. Initialize and Train
        model = DummyClassifier(strategy="stratified", random_state=42)
        model.fit(X_train, y_train)
        with open(model_path, "wb") as f:
            pickle.dump(model, f)
        print("Model trained and saved to pickle")

Model trained and saved to pickle


In [4]:
y_pred = model.predict(X_test)

In [5]:
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

In [6]:
# 4. Calculate Business Metrics
biz_metrics = calculate_business_metrics(y_test, y_pred, test_holidays)

In [7]:
mlflow.log_param("model_type", "DummyClassifier")
mlflow.log_param("strategy", "stratified")

mlflow.log_metric("accuracy", acc)
mlflow.log_metric("f1_weighted", f1)
mlflow.log_metric("holiday_accuracy", biz_metrics["holiday_accuracy"])
mlflow.log_metric("weighted_classification_error", biz_metrics["weighted_classification_error"])

In [8]:
# 6. Save Model Artifact
mlflow.log_artifact(model_path)

mlflow.sklearn.log_model(model, artifact_path="model")

print(f"Baseline Accuracy: {acc:.4f}")
print(f"Holiday Accuracy: {biz_metrics['holiday_accuracy']:.4f}")
print(f"Weighted Error: {biz_metrics['weighted_classification_error']:.4f}")

2026/05/01 20:20:49 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Baseline Accuracy: 0.4990
Holiday Accuracy: 0.5016
Weighted Error: 0.4989
